# ACDS smoke recovery on Kaggle

Resume the existing **ResEnc-M, outer fold 0, seed 7 smoke study** from
**ACDS Fullres GPU Smoke, version 2**. The source release is 0.3.1.
Completed trials are verified and retained; the registered splits, losses,
training budgets, initialization rules and evaluation policy are unchanged.
Smoke tests the pipeline and produces no outer-fold accuracy result.

1. Open the saved recovery version's **Output** tab. Under **Output actions**,
   choose **Create notebook** to attach that exact output version. A private
   recovery editor named **ACDS GPU Smoke Resume** was already created.
2. Import this notebook into that editor. Confirm the attached input is
   **ACDS Fullres GPU Smoke, version 2**. Use **GPU T4 x2**, internet enabled,
   and the original pinned Kaggle environment dated **2026-10-03**.
   Stop the old interactive session after confirming its recovery version.
3. Select **Save Version → Save & Run All**, and keep the notebook private.
   This starts a separate clean background session. Follow its **Logs** and
   **Output** tabs. The toolbar's interactive Run All is not the intended launch.

Recovery version (verified):
https://www.kaggle.com/code/asimraza138/acds-fullres-gpu-smoke/output?scriptVersionId=356717389

The first cell inventories the attached backup. Restore verifies every copied
file and each completed checkpoint's saved SHA256 before training. Files return
to their original absolute study paths. Public raw CTs are downloaded with the
existing checksum check and their staged links are repaired by the original
source. Preprocessing arrays and fitted evidence are preserved.

The driver has a two-hour soft session limit and a three-GiB free-space reserve.
A separate capacity check includes the remaining full checkpoints and temporary
saves; it stops before training if storage is insufficient. These are session
controls, not changes to the scientific protocol. No duplicate ZIP is created;
Kaggle saves the physical working files with the background version. The full
research matrix remains pending until its timing and larger storage are settled.


In [1]:
from pathlib import Path
import json, hashlib, shutil
BACKUP = Path('/kaggle/input/notebooks/asimraza138/acds-fullres-gpu-smoke')
SAVED_STUDY = BACKUP/'acds-persistent/resenc_m-smoke'
EXPECTED_PROTOCOL_SHA256 = '6759ffbac28b167529cd15de0f6f27067eb432443f4c1e0bb575ac436285ac24'

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8*1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

def protocol_sha256(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

assert (SAVED_STUDY/'protocol.json').is_file(), 'Attach ACDS Fullres GPU Smoke version 2 output'
saved_protocol = json.loads((SAVED_STUDY/'protocol.json').read_text())
assert protocol_sha256(saved_protocol) == EXPECTED_PROTOCOL_SHA256, 'The attached study protocol differs'
assert saved_protocol['profile'] == 'smoke' and saved_protocol['architecture'] == 'resenc_m'
assert saved_protocol['seeds'] == [7]
backup_files = sorted(p for p in SAVED_STUDY.rglob('*') if p.is_file())
checkpoint_files = sorted(SAVED_STUDY.glob('runs/**/checkpoint_*.pth'))
complete_files = sorted(SAVED_STUDY.glob('runs/**/complete.json'))
validated_files = sorted(SAVED_STUDY.glob('runs/**/inner_scores.json'))
assert checkpoint_files and complete_files, 'The attached version has no retained trained checkpoint'
by_top = {}
for path in backup_files:
    key = path.relative_to(SAVED_STUDY).parts[0]
    by_top[key] = by_top.get(key, 0) + path.stat().st_size
backup_inventory = {
    'backup': str(BACKUP), 'protocol_sha256': protocol_sha256(saved_protocol),
    'state': json.loads((SAVED_STUDY/'session/state.json').read_text()),
    'study_gib': sum(p.stat().st_size for p in backup_files)/1024**3,
    'by_top_gib': {k:v/1024**3 for k,v in by_top.items()},
    'checkpoint_count': len(checkpoint_files),
    'checkpoint_gib': sum(p.stat().st_size for p in checkpoint_files)/1024**3,
    'trained': [str(p.relative_to(SAVED_STUDY)) for p in complete_files],
    'validated': [str(p.relative_to(SAVED_STUDY)) for p in validated_files]
}
print('ACDS_BACKUP_INVENTORY', json.dumps(backup_inventory, indent=2), flush=True)


ACDS_BACKUP_INVENTORY {
  "backup": "/kaggle/input/notebooks/asimraza138/acds-fullres-gpu-smoke",
  "protocol_sha256": "6759ffbac28b167529cd15de0f6f27067eb432443f4c1e0bb575ac436285ac24",
  "state": {
    "started_at": "2026-10-09T08:41:01.767177+00:00",
    "status": "interrupted",
    "folds_requested": [
      0
    ],
    "profile": "smoke",
    "registered_budget_changed": false,
    "outer_scoring_started": false,
    "commands_completed_this_session": 12,
    "updated_at": "2026-10-09T09:49:41.963492+00:00",
    "elapsed_seconds": 4120.196339324,
    "action": "run-fold",
    "fold": 0,
    "phase": "search",
    "last_result": {
      "status": "paused",
      "phase": "search",
      "fold": 0,
      "seed": 7,
      "arm": "cavity",
      "trial": 1,
      "instruction": "Rerun the same command; the registered full budget is unchanged."
    },
    "reason": "user_interrupt; resume from last atomic checkpoint"
  },
  "study_gib": 8.43294015713036,
  "by_top_gib": {
    "nnUNet_

In [2]:
from pathlib import Path
import os, sys, subprocess, json, zipfile, shutil, hashlib
SYSTEM_PYTHON = shutil.which('python3.12')
assert SYSTEM_PYTHON, 'Python 3.12 is required for the pinned study environment'
TRAIN_ENV = Path('/kaggle/temp/acds-python312')
PYTHON = str(TRAIN_ENV/'bin/python')
if not Path(PYTHON).is_file():
    subprocess.check_call([SYSTEM_PYTHON,'-m','venv','--without-pip',str(TRAIN_ENV)])
if subprocess.run([PYTHON,'-m','pip','--version'],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL).returncode != 0:
    import urllib.request
    PIP_BOOTSTRAP = TRAIN_ENV/'get-pip.py'
    urllib.request.urlretrieve('https://bootstrap.pypa.io/get-pip.py',PIP_BOOTSTRAP)
    subprocess.check_call([PYTHON,str(PIP_BOOTSTRAP)])
training_version = json.loads(subprocess.check_output([PYTHON,'-c','import sys,json; print(json.dumps(list(sys.version_info[:2])))'],text=True))
assert training_version == [3,12], training_version
print({'kernel_python':sys.version.split()[0],'training_python':training_version,'interpreter':PYTHON})
IS_KAGGLE = Path('/kaggle').exists()
assert IS_KAGGLE, 'This configured notebook uses the attached Kaggle input'
BASE = Path('/kaggle/working')
INPUT_PROJECT = BACKUP/'acds-project/acds-controlled-study'
PROJECT_PARENT = BASE / 'acds-project'
PROJECT = PROJECT_PARENT / 'acds-controlled-study'
if not PROJECT.exists():
    assert (INPUT_PROJECT/'acds/fullres/cli.py').is_file(), 'Attach the private ACDS Project dataset'
    PROJECT_PARENT.mkdir(parents=True, exist_ok=True)
    shutil.copytree(INPUT_PROJECT, PROJECT)
assert (PROJECT/'acds/fullres/cli.py').is_file()
source_manifest = json.loads((PROJECT/'SOURCE_MANIFEST.json').read_text())
for relative_path, expected_hash in source_manifest.items():
    source_file = PROJECT / relative_path
    assert source_file.is_file(), f'Missing source file: {relative_path}'
    assert hashlib.sha256(source_file.read_bytes()).hexdigest() == expected_hash, f'Source mismatch: {relative_path}'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['nnUNet_compile'] = 'false'
os.environ['MPLBACKEND'] = 'Agg'
subprocess.run(['nvidia-smi'], check=True)
print('Verified source files:', len(source_manifest), 'Project:', PROJECT)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 28.9 MB/s  0:00:00
{'kernel_python': '3.13.15', 'training_python': [3, 12], 'interpreter': '/kaggle/temp/acds-python312/bin/python'}
Fri Oct  9 14:43:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W | 

Install the tested version family. The official PyTorch CUDA 12.4 command
is pinned below. If the runtime loaded Torch/Numpy before installation,
restart the notebook runtime, then rerun from the configuration cell.
Dependencies are pinned for reproducibility rather than current-version claims.

In [3]:
subprocess.check_call([PYTHON,'-m','pip','install','torch==2.6.0','torchvision==0.21.0',
                       '--index-url','https://download.pytorch.org/whl/cu124'])
subprocess.check_call([PYTHON,'-m','pip','install','-r',str(PROJECT/'requirements-fullres.txt')])
subprocess.check_call([PYTHON,'-m','pip','install','-e',str(PROJECT),'--no-deps'])

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 128.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 228.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 149.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 42.2 MB/s  0:00:07
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 105.4 MB/s  0:00:03
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 134.1 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 133.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 133.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 133.2 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.1/150.1 MB 140.1 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 134.9 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 M

0

In [4]:
os.environ['MPLBACKEND']='Agg'
VERIFY_SCRIPT = "import torch, numpy, scipy, importlib.metadata as metadata\nassert torch.__version__.split('+')[0]=='2.6.0', 'Restart after installing the pinned Torch'\nassert torch.cuda.is_available(), 'Enable an NVIDIA GPU runtime'\nprint({'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),\n       'nnunetv2':metadata.version('nnunetv2'),'numpy':numpy.__version__,'scipy':scipy.__version__})\n"
subprocess.check_call([PYTHON,"-c",VERIFY_SCRIPT],cwd=PROJECT)
subprocess.check_call([PYTHON,'-m','pytest','-q','tests/test_fullres_losses.py',
                       'tests/test_fullres_protocol.py','tests/test_fullres_evaluation.py',
                       'tests/test_fullres_selection.py','tests/test_fullres_trainer.py',
                       'tests/test_fullres_execution.py','tests/test_nnunet_datasets.py'],cwd=PROJECT)

{'torch': '2.6.0+cu124', 'cuda': '12.4', 'gpu': 'Tesla T4', 'nnunetv2': '2.6.2', 'numpy': '2.3.5', 'scipy': '1.16.1'}
...........................................                              [100%]
=============================== warnings summary ===============================
tests/test_fullres_evaluation.py::test_single_component_prior_and_reference_matching_are_distinct
  /kaggle/temp/acds-python312/lib/python3.12/site-packages/surface_distance/metrics.py:240: DeprecationWarning: Please import `correlate` from the `scipy.ndimage` namespace; the `scipy.ndimage.filters` namespace is deprecated and will be removed in SciPy 2.0.0.
    neighbour_code_map_gt = ndimage.filters.correlate(

tests/test_fullres_evaluation.py::test_single_component_prior_and_reference_matching_are_distinct
  /kaggle/temp/acds-python312/lib/python3.12/site-packages/surface_distance/metrics.py:242: DeprecationWarning: Please import `correlate` from the `scipy.ndimage` namespace; the `scipy.ndimage.filters` names

0

Choose one profile and persistent study path. On Colab, mount your own Drive
and use a folder with enough available space. On Kaggle, checkpoints in
`/kaggle/working` survive only when saved as notebook output; download/archive
them before the session ends and restore to exactly the same path.
The full matrix will need more storage than many free accounts provide.
`smoke` and `research` must use different study directories. A smoke result
cannot substitute for a completed research run.

In [5]:
PROFILE = 'smoke'       # Switch to 'research' only in a new directory.
ARCHITECTURE = 'resenc_m' # 'plain' is a separate, smaller baseline experiment.
FOLD = 0
FOLDS = [0] if PROFILE == 'smoke' else [0,1,2,3,4]
MAX_SESSION_HOURS = 2  # Soft limit; an active epoch/inference finishes first.
SESSION_EPOCHS = 1     # Save/reload at these epoch boundaries; full LR budget is unchanged.
MIN_FREE_GIB = 3       # Reserve for temporary atomic saves; not a full-matrix storage estimate.
AUTO_SNAPSHOT = False
if not IS_KAGGLE:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSISTENT = Path('/content/drive/MyDrive/ACDS')
else:
    PERSISTENT = BASE / 'acds-persistent'
STUDY = PERSISTENT / f'{ARCHITECTURE}-{PROFILE}'
DATA = Path('/kaggle/temp/acds-data')
STUDY.mkdir(parents=True,exist_ok=True)
def command(*args):
    subprocess.check_call([PYTHON,'-u','-m','acds.fullres.cli',*map(str,args)],cwd=PROJECT)
def module_command(module,*args):
    subprocess.check_call([PYTHON,'-u','-m',module,*map(str,args)],cwd=PROJECT)
SNAPSHOT = BASE / f'ACDS_{ARCHITECTURE}_{PROFILE}_resume.zip'
print('Study:',STUDY,'Free GiB:',round(shutil.disk_usage(STUDY).free/1024**3,2))

Study: /kaggle/working/acds-persistent/resenc_m-smoke Free GiB: 19.47


Restore the attached recovery output before preparation. The original
source repairs raw staging links, verifies fitted evidence, and resumes the
same registered model budgets. No research profile is initialized here.


In [6]:
# Restore physical files to the ORIGINAL absolute study path.
# Public native CT arrays are rebuilt from the checksum-verified archive later.
receipt_path = BASE/'acds_recovery_receipt.json'
GIB = 1024**3

def include_backup_file(path):
    relative = path.relative_to(SAVED_STUDY)
    if relative.name.endswith('.partial'):
        return False
    if relative.parts[0] == 'nnUNet_raw' and any(x in ('imagesTr', 'labelsTr') for x in relative.parts):
        return False
    return True

restore_files = [path for path in backup_files if include_backup_file(path)]
target_models = (len(saved_protocol['learning_rate_candidates'])
                 + (len(saved_protocol['arms'])-1)*len(saved_protocol['weights'])
                 + len(saved_protocol['seeds'])*len(saved_protocol['arms']))
one_checkpoint_bytes = max(path.stat().st_size for path in checkpoint_files)
restore_bytes = sum(path.stat().st_size for path in restore_files)
checkpoint_bytes = sum(path.stat().st_size for path in checkpoint_files)
retained_overhead = max(0, restore_bytes-checkpoint_bytes)
capacity_bytes = retained_overhead + target_models*one_checkpoint_bytes + 2*one_checkpoint_bytes + GIB
current_study_bytes = sum(p.stat().st_size for p in STUDY.rglob('*') if p.is_file())
usable_bytes = shutil.disk_usage(BASE).free + current_study_bytes
print('ACDS_RECOVERY_CAPACITY', json.dumps({
    'target_models': target_models, 'one_actual_checkpoint_gib': one_checkpoint_bytes/GIB,
    'restore_gib': restore_bytes/GIB, 'capacity_with_temporary_headroom_gib': capacity_bytes/GIB,
    'usable_gib': usable_bytes/GIB, 'runner_reserve_gib': MIN_FREE_GIB
}, indent=2), flush=True)
if usable_bytes < capacity_bytes:
    raise RuntimeError('Working storage cannot retain this smoke fold plus temporary saves; preserve the attached backup and provide more space')

if receipt_path.is_file():
    receipt = json.loads(receipt_path.read_text())
    assert receipt['study_path'] == str(STUDY)
    assert receipt['protocol_sha256'] == EXPECTED_PROTOCOL_SHA256
    assert protocol_sha256(json.loads((STUDY/'protocol.json').read_text())) == EXPECTED_PROTOCOL_SHA256
    print('Using the already-restored study; current checkpoints will not be overwritten.', flush=True)
else:
    if any(path.is_file() for path in STUDY.rglob('*')):
        raise RuntimeError('Nonempty study without a recovery receipt. Use a fresh Save & Run All session; do not overwrite existing checkpoints')
    copied_hashes = {}
    for index, source in enumerate(restore_files, 1):
        relative = source.relative_to(SAVED_STUDY)
        target = STUDY/relative
        target.parent.mkdir(parents=True, exist_ok=True)
        source_hash = file_sha256(source)
        shutil.copy2(source, target)
        if file_sha256(target) != source_hash:
            raise ValueError(f'Recovery copy hash differs: {relative}')
        copied_hashes[str(relative)] = source_hash
        if source.suffix == '.pth' or index % 100 == 0:
            print(f'Verified recovery file {index}/{len(restore_files)}: {relative}', flush=True)
    # Completed-run hashes bind retained model bytes to the original evidence.
    for marker in complete_files:
        completed = json.loads(marker.read_text())
        assert completed['protocol_sha256'] == EXPECTED_PROTOCOL_SHA256
        original_checkpoint = Path(completed['checkpoint'])
        relative_checkpoint = original_checkpoint.relative_to(STUDY)
        if file_sha256(STUDY/relative_checkpoint) != completed['checkpoint_sha256']:
            raise ValueError(f'Completed checkpoint changed: {relative_checkpoint}')
    receipt = {
        'study_path': str(STUDY), 'backup_input': str(BACKUP),
        'protocol_sha256': EXPECTED_PROTOCOL_SHA256,
        'verified_copied_files': copied_hashes,
        'restored_trained_models': len(complete_files),
        'restored_validated_models': len(validated_files),
        'registered_budget_changed': False, 'raw_staging_rebuilt_from_verified_archive': True
    }
    receipt_path.write_text(json.dumps(receipt, indent=2))
    (BASE/'acds_backup_inventory.json').write_text(json.dumps(backup_inventory, indent=2))
    print('ACDS_RECOVERY_VERIFIED', json.dumps({k:v for k,v in receipt.items() if k != 'verified_copied_files'}, indent=2), flush=True)


ACDS_RECOVERY_CAPACITY {
  "target_models": 18,
  "one_actual_checkpoint_gib": 0.7592343296855688,
  "restore_gib": 8.43294015713036,
  "capacity_with_temporary_headroom_gib": 17.02528345398605,
  "usable_gib": 19.473987579345703,
  "runner_reserve_gib": 3
}
Verified recovery file 100/360: nnUNet_preprocessed/Dataset950_ACDSSpleenOuter0/nnUNetPlans_3d_fullres/spleen_60_seg.b2nd
Verified recovery file 120/360: runs/search/outer_0/seed_7/baseline/trial_0/Dataset950_ACDSSpleenOuter0/nnUNetTrainerACDSFullres__nnUNetResEncUNetMPlans__3d_fullres/fold_0/checkpoint_final.pth
Verified recovery file 142/360: runs/search/outer_0/seed_7/baseline/trial_1/Dataset950_ACDSSpleenOuter0/nnUNetTrainerACDSFullres__nnUNetResEncUNetMPlans__3d_fullres/fold_0/checkpoint_final.pth
Verified recovery file 164/360: runs/search/outer_0/seed_7/boundary_distance/trial_0/Dataset950_ACDSSpleenOuter0/nnUNetTrainerACDSFullres__nnUNetResEncUNetMPlans__3d_fullres/fold_0/checkpoint_final.pth
Verified recovery file 186/360:

In [7]:
RESTORE_ZIP = None  # Example: Path('/kaggle/input/acds-resume/ACDS_resume.zip')
if RESTORE_ZIP is not None:
    module_command('acds.fullres.snapshots','restore','--root',STUDY,'--archive',RESTORE_ZIP)
# The public archive is downloaded with its known byte count and MD5/SHA256 audit.
RAW_POINTER = BASE/'acds_raw_path.json'
DOWNLOAD_SCRIPT = 'from acds.spleen.download import download; from pathlib import Path; import sys,json; raw=download(Path(sys.argv[1])); Path(sys.argv[2]).write_text(json.dumps(str(raw)))'
subprocess.check_call([PYTHON,'-c',DOWNLOAD_SCRIPT,str(DATA),str(RAW_POINTER)],cwd=PROJECT)
RAW = Path(json.loads(RAW_POINTER.read_text()))
command('init','--previous',PROJECT/'configs/controlled_protocol.json','--root',STUDY,
        '--profile',PROFILE,'--architecture',ARCHITECTURE)
command('prepare','--root',STUDY,'--raw',RAW,'--fold',FOLD)
command('resources','--root',STUDY,'--fold',FOLD)

Downloaded 128 MiB
Downloaded 256 MiB
Downloaded 384 MiB
Downloaded 512 MiB
Downloaded 640 MiB
Downloaded 768 MiB
Downloaded 896 MiB
Downloaded 1024 MiB
Downloaded 1152 MiB
Downloaded 1280 MiB
Downloaded 1408 MiB
{
  "protocol_sha256": "6759ffbac28b167529cd15de0f6f27067eb432443f4c1e0bb575ac436285ac24",
  "search_runs": 60,
  "final_runs": 30,
  "search_updates": 240,
  "final_updates": 120,
  "baseline_note": "Three baseline LR trials precede three loss-weight trials per constraint; the selected LR is shared.",
  "runtime_note": "Measure GPU seconds per update before committing the full allocation."
}
{
  "prepared_fold": 0,
  "outer_patients_staged": false,
  "plans_fitted_on_training_only": true
}
nnUNet_raw is not defined and nnU-Net can only be used on data for which preprocessed files are already present on your system. nnU-Net cannot be used for experiment planning and preprocessing like this. If this is not intended, please read documentation/setting_up_paths.md for information 

The session runner automatically advances preparation, baseline optimization,
individual constraint search, locked selection and final training for FOLDS.
One launch runs until the soft session limit, then saves resumable state.
Rerun the same cell in the next GPU session; no manual phase switching is
required. Source/environment/split checks stay active in the scientific CLI.

`smoke` runs only fold 0 and does not read any outer labels. `research`
progresses all five folds and freezes the whole matrix before outer scoring
and report generation. Training stops on failures rather than advancing.
This process needs a live GPU session; it cannot keep running after that
session is terminated. The automatic ZIP needs room for a second full copy.

In [8]:
module_command('acds.fullres.execution','doctor','--device','cuda',
               '--output',STUDY/'session/compute.json')
try:
    module_command('acds.fullres.execution','run','--root',STUDY,'--raw',RAW,
                   '--folds',*FOLDS,'--max-hours',MAX_SESSION_HOURS,
                   '--epoch-chunk',SESSION_EPOCHS,'--min-free-gib',MIN_FREE_GIB,
                   '--device','cuda','--through','report')
finally:
    state_path = STUDY/'session/state.json'
    progress = {
        'state': json.loads(state_path.read_text()) if state_path.is_file() else None,
        'trained_models': len(list(STUDY.glob('runs/**/complete.json'))),
        'validated_search_models': len(list(STUDY.glob('runs/**/inner_scores.json'))),
        'checkpoint_count': len(list(STUDY.glob('runs/**/checkpoint_*.pth'))),
        'free_gib': shutil.disk_usage(STUDY).free/1024**3,
        'scientific_accuracy_experiment': False
    }
    (BASE/'acds_smoke_progress.json').write_text(json.dumps(progress, indent=2))
    print('ACDS_SMOKE_PROGRESS', json.dumps(progress, indent=2), flush=True)
command('status','--root',STUDY)


{
  "checked_at": "2026-10-09T14:53:42.218913+00:00",
  "python": "3.12.3",
  "device_requested": "cuda",
  "cuda_available": true,
  "ready": true,
  "issues": [],
  "packages": {
    "nnunetv2": "2.6.2",
    "dynamic-network-architectures": "0.4.1",
    "acvl-utils": "0.2.3",
    "batchgenerators": "0.25.1",
    "batchgeneratorsv2": "0.3.0",
    "blosc2": "3.5.1",
    "numpy": "2.3.5",
    "scipy": "1.16.1",
    "scikit-image": "0.25.2",
    "nibabel": "5.3.2",
    "SimpleITK": "2.5.6",
    "surface-distance": "0.1"
  },
  "torch": "2.6.0+cu124",
  "torch_cuda": "12.4",
  "gpu": "Tesla T4",
  "gpu_memory_GiB": 14.56219482421875
}
[2026-10-09T14:53:46.591863+00:00] prepare fold=0 phase=None
{
  "prepared_fold": 0,
  "outer_patients_staged": false,
  "plans_fitted_on_training_only": true
}
[2026-10-09T14:53:53.874823+00:00] resources fold=0 phase=None
nnUNet_raw is not defined and nnU-Net can only be used on data for which preprocessed files are already present on your system. nnU-Net 

Review measured resources after each session. These are actual saved epoch
timing/memory observations, not promised runtime. Selection and final runs
are handled by the runner above without changing any registered budgets.

In [9]:
for fold in FOLDS:
    if (STUDY/'preparation'/f'outer_{fold}.json').exists():
        command('resources','--root',STUDY,'--fold',fold)

nnUNet_raw is not defined and nnU-Net can only be used on data for which preprocessed files are already present on your system. nnU-Net cannot be used for experiment planning and preprocessing like this. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up properly.
nnUNet_preprocessed is not defined and nnU-Net can not be used for preprocessing or training. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up.
nnUNet_results is not defined and nnU-Net cannot be used for training or inference. If this is not intended behavior, please read documentation/setting_up_paths.md for information on how to set this up.
{
  "fold": 0,
  "architecture": "resenc_m",
  "parameters": 101857674,
  "batch_size": 2,
  "patch_size": [
    56,
    192,
    160
  ],
  "spacing_mm": [
    1.6900090336799622,
    0.7949219942092896,
    0.7949219942092896
  ],
  "one_full_SGD_checkpoint_lower_boun

The background version saves the physical study directory as output.
No second ZIP is written because it would duplicate large checkpoints.
Preserve this completed or paused version and attach its output for another
session. Check `acds_smoke_progress.json` and `session/state.json` for status.


In [10]:
if AUTO_SNAPSHOT:
    module_command('acds.fullres.snapshots','create','--root',STUDY,'--output',SNAPSHOT)
    print('Save/download this file before the session ends:',SNAPSHOT)
else:
    print('Automatic snapshot disabled to avoid duplicating retained checkpoints. Study:', STUDY)


Automatic snapshot disabled to avoid duplicating retained checkpoints. Study: /kaggle/working/acds-persistent/resenc_m-smoke


The runner creates this report only after **every** final fold/seed/arm
has completed. Freeze checks
epoch-zero weights, optimizer initialization, every image/target batch hash,
source/protocol/checkpoint hashes and the paired execution environment.
Native evaluation then reports raw, largest-component and shared hole-fill
policies; missing files/empty predictions stay visible. The JSON and HTML
report contain descriptive patient-paired effects. They are not independent
confirmation because these development patients were inspected previously.

In [11]:
from IPython.display import FileLink, display
if PROFILE == 'research' and (STUDY/'evaluation/report.html').is_file():
    display(FileLink(str(STUDY/'evaluation/report.html')))
    display(FileLink(str(STUDY/'evaluation/results.json')))
else:
    print('No research report yet. Continue the runner; smoke never scores outer labels.')

No research report yet. Continue the runner; smoke never scores outer labels.


Export a baseline fold ensemble for deployment. Each of its 5 models has
verified full/portable tensor equality. The source pipeline also implements
`ensemble` for unseen, single-channel CTs. It rejects development identifiers
and renamed known CT files. A full-fold ensemble is never used to score a
patient whose label entered another member's training set.

Native GPU equivalence between full and portable archives still needs a
representative saved-prediction check before a clinical/deployment claim.
If a constraint arm is later selected using these outer results, describe
that as development selection and obtain a genuinely new evaluation set.

In [12]:
EXPORT = PERSISTENT / f'{ARCHITECTURE}-{PROFILE}-baseline-seed7-inference'
if PROFILE == 'research' and (STUDY/'evaluation/report.html').is_file():
    command('export','--root',STUDY,'--output',EXPORT,'--arm','baseline','--seed',7)
    display(FileLink(str(EXPORT.with_suffix('.zip'))))
else:
    print('Complete the registered research matrix before exporting inference models.')
# Unseen CT filenames: patient_001_0000.nii.gz in an input directory.
# command('ensemble','--root',STUDY,'--known-raw',RAW,'--images',BASE/'unseen_cts',
#         '--output',PERSISTENT/'unseen_predictions','--arm','baseline','--seed',7,'--device','cuda')

Complete the registered research matrix before exporting inference models.
